# 12 — Opportunity loss and exploratory mobility linkage

This notebook has two purposes:

1. document the current spatial mismatch between the local thermal-accessibility PoC and the citywide mobile-phone data;
2. reconstruct the exact transit-stop opportunities from Notebook 07 and examine which origin–stop opportunities are lost under thermal impedance.

The district comparison remains exploratory because the 49 accessibility origins occupy only a small part of Sevilla.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd

from shapely.geometry import box

ACCESS_DIR = OUTPUTS / "accessibility"
MOBILE_INTERIM = PROJECT_ROOT / "data" / "mobile" / "interim"
MOBILE_PROCESSED = PROJECT_ROOT / "data" / "mobile" / "processed"

access = pd.read_csv(
    ACCESS_DIR / "origin_accessibility_lambda_005.csv"
)
access["origin"] = access["origin"].astype(str)

mobility = pd.read_parquet(
    MOBILE_PROCESSED / "sevilla_district_hot_reference_mobility.parquet"
)

zones = gpd.read_file(
    MOBILE_INTERIM / "sevilla_districts_v2.gpkg"
)

## 1. Check the spatial overlap with Sevilla districts

In [ ]:
G_display = ox.load_graphml(
    DATA_INTERIM / "sevilla_walk_thermal.graphml"
)

nodes_display, _ = ox.graph_to_gdfs(G_display)
nodes_display = nodes_display.reset_index()
nodes_display["osmid"] = nodes_display["osmid"].astype(str)

access_origins = (
    nodes_display[
        nodes_display["osmid"].isin(access["origin"])
    ]
    .merge(
        access,
        left_on="osmid",
        right_on="origin",
        how="inner",
    )
)

zones_projected = zones.to_crs(access_origins.crs)

origins_district = gpd.sjoin(
    access_origins,
    zones_projected[["id", "name", "geometry"]],
    how="left",
    predicate="within",
)

print("Origins assigned:", origins_district["id"].notna().sum(), "/", len(origins_district))
display(origins_district["name"].value_counts(dropna=False))

### Interpretation

The current thermal PoC does not represent all 11 Sevilla districts. A citywide accessibility–mobility validation would therefore be misleading at this stage. The opportunity-level analysis below is used instead to explain the internal accessibility mechanism.

## 2. Reconstruct the exact cleaned graph and stop-node set

In [ ]:
G_raw = ox.load_graphml(
    DATA_INTERIM / "sevilla_walk_thermal.graphml"
)

numeric_fields = [
    "length",
    "walk_time_s",
    "utci_max",
    "heat_excess",
    "thermal_cost_s",
    "cost_ratio",
]

for _, _, _, data in G_raw.edges(keys=True, data=True):
    for field in numeric_fields:
        if field in data:
            try:
                data[field] = float(data[field])
            except (TypeError, ValueError):
                data[field] = np.nan

largest_component = max(
    nx.strongly_connected_components(G_raw),
    key=len,
)

G_preclean = G_raw.subgraph(largest_component).copy()

In [ ]:
nodes_pre, edges_pre = ox.graph_to_gdfs(G_preclean)

thermal_polygon = gpd.GeoSeries(
    [box(*edges_pre.total_bounds)],
    crs=edges_pre.crs,
).to_crs(4326).iloc[0]

stops = ox.features_from_polygon(
    thermal_polygon,
    tags={
        "highway": "bus_stop",
        "public_transport": ["platform", "stop_position"],
    },
)

stops = stops[
    stops.geometry.geom_type.isin(
        ["Point", "Polygon", "MultiPolygon"]
    )
].copy()

stops = stops.to_crs(G_preclean.graph["crs"])
stops["geometry"] = stops.geometry.centroid
stops = stops.drop_duplicates(subset="geometry")

stops["node"] = ox.distance.nearest_nodes(
    G_preclean,
    X=stops.geometry.x.values,
    Y=stops.geometry.y.values,
)

stop_nodes_preclean = list(pd.unique(stops["node"]))

In [ ]:
nodes_all, edges_all = ox.graph_to_gdfs(G_preclean)

edges_valid = edges_all[
    edges_all["walk_time_s"].notna()
    & edges_all["utci_max"].notna()
    & edges_all["heat_excess"].notna()
].copy()

valid_node_ids = (
    set(edges_valid.index.get_level_values("u"))
    | set(edges_valid.index.get_level_values("v"))
)

nodes_valid = nodes_all.loc[
    nodes_all.index.intersection(valid_node_ids)
].copy()

nodes_valid["x"] = nodes_valid.geometry.x
nodes_valid["y"] = nodes_valid.geometry.y

G_valid = ox.graph_from_gdfs(
    nodes_valid,
    edges_valid,
    graph_attrs={"crs": G_preclean.graph["crs"]},
)

largest_component = max(
    nx.strongly_connected_components(G_valid),
    key=len,
)

G_main = G_valid.subgraph(largest_component).copy()

stop_nodes = [
    node
    for node in stop_nodes_preclean
    if node in G_main
]

# Recreate λ = 0.05 exactly.
for _, _, _, data in G_main.edges(keys=True, data=True):
    walk_time = float(data["walk_time_s"])
    heat_excess = float(data["heat_excess"])
    data["thermal_0.05"] = walk_time * (1 + 0.05 * heat_excess)

print("Final nodes:", G_main.number_of_nodes())
print("Final edges:", G_main.number_of_edges())
print("Stop nodes:", len(stop_nodes))

## 3. Validate the reconstructed accessibility calculation

In [ ]:
graph_node_lookup = {
    str(node): node
    for node in G_main.nodes
}

rows = []

for _, row in access.iterrows():
    origin_str = str(row["origin"])
    origin = graph_node_lookup[origin_str]

    normal_costs = nx.single_source_dijkstra_path_length(
        G_main,
        origin,
        cutoff=600,
        weight="walk_time_s",
    )

    thermal_costs = nx.single_source_dijkstra_path_length(
        G_main,
        origin,
        cutoff=600,
        weight="thermal_0.05",
    )

    normal_stops = {
        stop for stop in stop_nodes
        if stop in normal_costs
    }

    thermal_stops = {
        stop for stop in stop_nodes
        if stop in thermal_costs
    }

    rows.append({
        "origin": origin_str,
        "normal_stop_count": len(normal_stops),
        "thermal_stop_count": len(thermal_stops),
    })

reconstructed = pd.DataFrame(rows)

check = access.merge(
    reconstructed,
    on="origin",
    how="left",
)

print(
    "Normal counts match:",
    (check["normal_access"] == check["normal_stop_count"]).all()
)

print(
    "Thermal counts match:",
    (check["thermal_access"] == check["thermal_stop_count"]).all()
)

## 4. Build the origin–stop opportunity table

In [ ]:
pair_rows = []

for _, row in access.iterrows():
    origin_str = str(row["origin"])
    origin = graph_node_lookup[origin_str]

    normal_costs = nx.single_source_dijkstra_path_length(
        G_main,
        origin,
        cutoff=600,
        weight="walk_time_s",
    )

    thermal_costs = nx.single_source_dijkstra_path_length(
        G_main,
        origin,
        cutoff=600,
        weight="thermal_0.05",
    )

    for stop in stop_nodes:
        if stop not in normal_costs:
            continue

        normal_cost = normal_costs[stop]

        pair_rows.append({
            "origin": origin_str,
            "stop_node": stop,
            "normal_cost_s": normal_cost,
            "margin_to_budget_s": 600 - normal_cost,
            "lost": int(stop not in thermal_costs),
        })

opportunity_pairs = pd.DataFrame(pair_rows)

print("Normally reachable origin–stop opportunities:", len(opportunity_pairs))
print("Opportunities lost:", opportunity_pairs["lost"].sum())
print(
    "Overall loss rate:",
    f"{100 * opportunity_pairs['lost'].mean():.1f}%"
)

## 5. Opportunity loss by normal walking-time band

In [ ]:
bins = [0, 120, 240, 360, 420, 480, 540, 600]
labels = [
    "0–2 min",
    "2–4 min",
    "4–6 min",
    "6–7 min",
    "7–8 min",
    "8–9 min",
    "9–10 min",
]

opportunity_pairs["normal_cost_band"] = pd.cut(
    opportunity_pairs["normal_cost_s"],
    bins=bins,
    labels=labels,
    include_lowest=True,
)

loss_by_band = (
    opportunity_pairs
    .groupby("normal_cost_band", observed=True)
    .agg(
        opportunities=("lost", "size"),
        lost_opportunities=("lost", "sum"),
        loss_rate=("lost", "mean"),
    )
    .reset_index()
)

loss_by_band["loss_rate_pct"] = (
    100 * loss_by_band["loss_rate"]
)

display(loss_by_band)

fig, ax = plt.subplots(figsize=(7, 5))

ax.bar(
    loss_by_band["normal_cost_band"].astype(str),
    loss_by_band["loss_rate_pct"],
)

ax.set_xlabel("Normal walking cost")
ax.set_ylabel("Thermally lost opportunities (%)")
ax.set_title("Thermal opportunity loss by normal walking-time band")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

## 6. Diagnose the effective cost multiplier

In [ ]:
ratios = pd.Series([
    float(data["thermal_0.05"]) / float(data["walk_time_s"])
    for _, _, _, data in G_main.edges(keys=True, data=True)
    if float(data["walk_time_s"]) > 0
])

display(ratios.describe())

mean_ratio = ratios.mean()
median_ratio = ratios.median()

print(
    f"Mean-ratio equivalent conventional threshold: "
    f"{600 / mean_ratio:.1f} s "
    f"({600 / mean_ratio / 60:.2f} min)"
)

print(
    f"Median-ratio equivalent conventional threshold: "
    f"{600 / median_ratio:.1f} s "
    f"({600 / median_ratio / 60:.2f} min)"
)

## Interpretation

The sharp loss of opportunities beyond roughly 6–7 minutes is primarily a **model-mechanism diagnostic**. Under the daily-maximum UTCI surface, the cost multiplier is close to 1.48 over much of the network, so a 600 s generalized budget behaves like roughly 406 s of conventional walking time.

This is **not** evidence for a universal behavioral 6–7 minute heat threshold. It shows that the current daily-max thermal field is too spatially saturated for the explanatory stage and motivates common-hour thermal surfaces in the next model version.